# Data Cleaning and Preprocessing

Dataset: [Most Streamed Spotify Songs 2023](https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023) (Kaggle).

In this notebook we load the raw data, deal with its problems and save a clean version to `spotify-2023-clean.csv`. The analysis lives in `notebooks/analysis.ipynb`.

## 1. Loading and first look

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/spotify-2023.csv", encoding="latin-1")
df.head()

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 953 entries, 0 to 952
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   track_name            953 non-null    str  
 1   artist(s)_name        953 non-null    str  
 2   artist_count          953 non-null    int64
 3   released_year         953 non-null    int64
 4   released_month        953 non-null    int64
 5   released_day          953 non-null    int64
 6   in_spotify_playlists  953 non-null    int64
 7   in_spotify_charts     953 non-null    int64
 8   streams               953 non-null    str  
 9   in_apple_playlists    953 non-null    int64
 10  in_apple_charts       953 non-null    int64
 11  in_deezer_playlists   953 non-null    str  
 12  in_deezer_charts      953 non-null    int64
 13  in_shazam_charts      903 non-null    str  
 14  bpm                   953 non-null    int64
 15  key                   858 non-null    str  
 16  mode               

## 2. Problems found during the first look

1. The columns `streams`, `in_deezer_playlists`, `in_shazam_charts` have type `str` even though they contain numbers - large values use commas as thousands separators (e.g. `"2,445"`);
2. the columns `key` and `in_shazam_charts` contain 95 and 50 missing values respectively;
3. one row has a chunk of descriptive text in `streams` instead of a number (we find it below);
4. track names with non-Latin characters got mangled when read with `latin-1` - this does not affect the numeric analysis, so we leave them as is.

In [4]:
# look for rows where streams cannot be converted to a number
df[pd.to_numeric(df["streams"], errors="coerce").isna()]["streams"]

574    BPM110KeyAModeMajorDanceability53Valence75Ener...
Name: streams, dtype: str

## 3. Duplicates

There are no fully duplicated rows, but the same track could appear twice with slightly different chart values - we check by the pair "track name + artist".

In [5]:
print("fully duplicated rows:", df.duplicated().sum())

dups = df[df.duplicated(subset=["track_name", "artist(s)_name"], keep=False)]
dups[["track_name", "artist(s)_name", "released_year", "streams"]].sort_values("track_name")

fully duplicated rows: 0


,track_name,artist(s)_name,released_year,streams
372,About Damn Time,Lizzo,2022,723894473
764,About Damn Time,Lizzo,2022,723894473
178,SNAP,Rosa Linn,2022,726307468
873,SNAP,Rosa Linn,2022,711366595
345,SPIT IN MY FACE!,ThxSoMch,2022,303216294
482,SPIT IN MY FACE!,ThxSoMch,2022,301869854
512,Take My Breath,The Weeknd,2021,130655803
616,Take My Breath,The Weeknd,2021,432702334


In [6]:
df = df.drop_duplicates(subset=["track_name", "artist(s)_name"], keep="first").reset_index(drop=True)
len(df)

949

## 4. Type casting

The row with the broken `streams` value cannot be recovered - there is simply no stream count in it, so we drop it. In `in_deezer_playlists` and `in_shazam_charts` we remove the thousands separators and convert the columns to numbers.

In [7]:
# streams: drop the broken row and convert to integers
df["streams"] = pd.to_numeric(df["streams"], errors="coerce")
df = df.dropna(subset=["streams"]).reset_index(drop=True)
df["streams"] = df["streams"].astype("int64")

# remove thousands separators and convert to numbers
for col in ["in_deezer_playlists", "in_shazam_charts"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", ""), errors="coerce")

df.dtypes[["streams", "in_deezer_playlists", "in_shazam_charts"]]

streams                  int64
in_deezer_playlists      int64
in_shazam_charts       float64
dtype: object

## 5. Missing values

- `key` - a categorical feature; "guessing" the key for a track would be incorrect, so we mark missing values as a separate category `Unknown`;
- `in_shazam_charts` - a missing value here effectively means the track did not enter the Shazam chart, so we fill it with zero.

In [8]:
df["key"] = df["key"].fillna("Unknown")
df["in_shazam_charts"] = df["in_shazam_charts"].fillna(0).astype("int64")

print("missing values left in the whole dataframe:", df.isna().sum().sum())

missing values left in the whole dataframe: 0


## 6. Outliers

In [9]:
df[["streams", "bpm", "artist_count", "released_year"]].describe().round(1)

,streams,bpm,artist_count,released_year
count,9.480000e+02,948.0,948.0,948.0
mean,5.140179e+08,122.5,1.6,2018.3
std,5.679277e+08,28.0,0.9,11.0
min,2.762000e+03,65.0,1.0,1930.0
25%,1.411439e+08,99.0,1.0,2020.0
50%,2.876903e+08,120.5,1.0,2022.0
75%,6.729425e+08,140.0,2.0,2022.0
max,3.703895e+09,206.0,8.0,2023.0


The `streams` distribution is heavily right-skewed: the median is around 0.3 billion, while the maximum is 3.7 billion. But these are not data errors - they are real mega-hits, and we must not drop them.

`bpm` lies within realistic musical bounds, no anomalies. Old tracks (the minimum release year is 1930) are also kept: this is classic music that still gains streams, and such observations are useful for analysing track age.

## 7. New features

We add derived features that will be useful in the analysis:

- `release_date` - a proper release date instead of three separate columns;
- `track_age_years` - the age of the track as of the dataset (year 2023);
- `total_playlists` - the total number of playlists across all platforms;
- `chart_platforms` - on how many of the four platforms the track entered the charts.

In [10]:
df["release_date"] = pd.to_datetime(
    dict(year=df["released_year"], month=df["released_month"], day=df["released_day"])
)

df["track_age_years"] = 2023 - df["released_year"]

df["total_playlists"] = (
    df["in_spotify_playlists"] + df["in_apple_playlists"] + df["in_deezer_playlists"]
)

chart_cols = ["in_spotify_charts", "in_apple_charts", "in_deezer_charts", "in_shazam_charts"]
df["chart_platforms"] = (df[chart_cols] > 0).sum(axis=1)

df[["track_name", "release_date", "track_age_years", "total_playlists", "chart_platforms"]].head()

,track_name,release_date,track_age_years,total_playlists,chart_platforms
0,Seven (feat. Latto) (Explicit Ver.),2023-07-14,0,641,4
1,LALA,2023-03-23,0,1580,4
2,vampire,2023-06-30,0,1582,4
3,Cruel Summer,2019-08-23,4,8099,4
4,WHERE SHE GOES,2023-05-18,0,3304,4


## 8. Enrichment via an external API (iTunes Search)

The source dataset has no track genre. We pull it from an external source - the public **iTunes Search API** (no key required). Using `track_name` + the first artist we look up the track and take the `primaryGenreName` field.

We cache the requests on disk (`data/itunes_cache.json`): on a re-run the notebook does not go to the network and is reproducible, even if the API is unavailable. Errors and not-found tracks are marked `Unknown`.

In [11]:
import time
import json
import requests
from pathlib import Path

CACHE_PATH = Path("../data/itunes_cache.json")
cache = json.loads(CACHE_PATH.read_text(encoding="utf-8")) if CACHE_PATH.exists() else {}


def fetch_genre(track, artist):
    """Return the track genre from the iTunes Search API, or None if not found or if the request fails."""
    primary_artist = artist.split(",")[0].strip()
    key = f"{track} {primary_artist}"
    if key in cache:
        return cache[key]
    try:
        resp = requests.get(
            "https://itunes.apple.com/search",
            params={"term": key, "entity": "song", "limit": 1},
            timeout=10,
        )
        resp.raise_for_status()
        results = resp.json().get("results", [])
        genre = results[0]["primaryGenreName"] if results else None
    except (requests.RequestException, KeyError, ValueError):
        genre = None
    cache[key] = genre  # store the result (hits and misses) so re-runs are reproducible and offline
    return genre


genres = []
for i, row in df.iterrows():
    # check the cache before the call so we only throttle real network requests, not cached lookups
    primary_artist = row["artist(s)_name"].split(",")[0].strip()
    was_cached = f"{row['track_name']} {primary_artist}" in cache
    genres.append(fetch_genre(row["track_name"], row["artist(s)_name"]))
    if i % 50 == 0:
        print(f"{i}/{len(df)}")
    if not was_cached:
        time.sleep(0.3)

df["genre"] = genres
CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")

matched = df["genre"].notna().sum()
print(f"found a genre for {matched} of {len(df)} tracks ({matched / len(df):.0%})")
df["genre"] = df["genre"].fillna("Unknown")
df["genre"].value_counts().head(10)

0/948
50/948
100/948
150/948
200/948
250/948
300/948
350/948
400/948


450/948
500/948
550/948
600/948
650/948
700/948
750/948
800/948
850/948
900/948
found a genre for 539 of 948 tracks (57%)


genre
Unknown            409
Pop                127
Hip-Hop/Rap         84
Alternative         45
R&B/Soul            42
Urbano latino       41
K-Pop               33
Música Mexicana     25
Dance               19
Latin               16
Name: count, dtype: int64

## 9. Saving the result

In [12]:
df.to_csv("../data/spotify-2023-clean.csv", index=False)
print(f"saved {df.shape[0]} rows and {df.shape[1]} columns")

saved 948 rows and 29 columns


## Cleaning summary

- removed duplicate tracks and one row with a broken `streams` value;
- `streams`, `in_deezer_playlists`, `in_shazam_charts` converted to numeric types (the source mixed in thousands separators);
- missing values handled: `key` -> category `Unknown`, `in_shazam_charts` -> 0;
- extreme stream values were checked and deliberately kept;
- added 4 derived features: `release_date`, `track_age_years`, `total_playlists`, `chart_platforms`;
- enriched the dataset with genre via an external API.

The clean dataset is saved to `data/spotify-2023-clean.csv`.